## Implementing PCA Step by Step

In [1]:
import numpy as np
import pandas as pd

np.random.seed(23) 

# Creating a dataframe of 40 rows and 4 columns

mu_vec1 = np.array([0,0,0])
cov_mat1 = np.array([[1,0,0],[0,1,0],[0,0,1]])
class1_sample = np.random.multivariate_normal(mu_vec1, cov_mat1, 20)

df = pd.DataFrame(class1_sample,columns=['feature1','feature2','feature3'])
df['target'] = 1

mu_vec2 = np.array([1,1,1])
cov_mat2 = np.array([[1,0,0],[0,1,0],[0,0,1]])
class2_sample = np.random.multivariate_normal(mu_vec2, cov_mat2, 20)

df1 = pd.DataFrame(class2_sample,columns=['feature1','feature2','feature3'])

df1['target'] = 0

df = pd.concat([df, df1], ignore_index=True)

df = df.sample(40)

In [2]:
df.head()

,feature1,feature2,feature3,target
2,-0.367548,-1.137460,-1.322148,1
34,0.177061,-0.598109,1.226512,0
14,0.420623,0.411620,-0.071324,1
11,1.968435,-0.547788,-0.679418,1
12,-2.506230,0.146960,0.606195,1


In [ ]:
# Plotting the dataframe in 3d 

import plotly.express as px

fig = px.scatter_3d(df, x=df['feature1'], y=df['feature2'], z=df['feature3'],
              color=df['target'].astype('str'))
fig.update_traces(marker=dict(size=12,
                              line=dict(width=2,
                                        color='DarkSlateGrey')),
                  selector=dict(mode='markers'))

fig.show()

In [4]:
# Step 1 - Apply standard scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

df.iloc[:,0:3] = scaler.fit_transform(df.iloc[:,0:3])

In [5]:
# Step 2 - Find Covariance Matrix
covariance_matrix = np.cov([df.iloc[:,0],df.iloc[:,1],df.iloc[:,2]])
print('Covariance Matrix:\n', covariance_matrix)

Covariance Matrix:
 [[1.02564103 0.20478114 0.080118  ]
 [0.20478114 1.02564103 0.19838882]
 [0.080118   0.19838882 1.02564103]]


In [6]:
# Step 3 - Finding EV and EVs
eigen_values, eigen_vectors = np.linalg.eig(covariance_matrix)

In [7]:
eigen_values

array([1.3536065 , 0.94557084, 0.77774573])

In [8]:
eigen_vectors

array([[-0.53875915, -0.69363291,  0.47813384],
       [-0.65608325, -0.01057596, -0.75461442],
       [-0.52848211,  0.72025103,  0.44938304]])

In [9]:
pc = eigen_vectors[0:2]
pc

array([[-0.53875915, -0.69363291,  0.47813384],
       [-0.65608325, -0.01057596, -0.75461442]])

In [10]:
transformed_df = np.dot(df.iloc[:,0:3],pc.T)
# (40,3) * (3,2) = (40,2)
new_df = pd.DataFrame(transformed_df,columns=['PC1','PC2'])
new_df['target'] = df['target'].values
new_df.head()

,PC1,PC2,target
0,0.599433,1.795862,1
1,1.056919,-0.212737,0
2,-0.271876,0.498222,1
3,-0.621586,0.023110,1
4,1.567286,1.730967,1


In [11]:
# Plotting the new dataframe after dimensionality is reduced from 3D to 2D
new_df['target'] = new_df['target'].astype('str')
fig = px.scatter(x=new_df['PC1'],
                 y=new_df['PC2'],
                 color=new_df['target'],
                 color_discrete_sequence=px.colors.qualitative.G10
                )

fig.update_traces(marker=dict(size=12,
                              line=dict(width=2,
                                        color='DarkSlateGrey')),
                  selector=dict(mode='markers'))
fig.show()

## Using Scikit Learn's implementation

In [12]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
pca.fit_transform(df.iloc[:,0:3],df.iloc[:,-1])

array([[-2.3028766 , -0.75759296],
       [-0.52446416,  0.52022134],
       [-0.38027543, -0.45944328],
       [-0.53426153, -1.77377341],
       [-1.61043442,  1.74781887],
       [ 0.85575955, -0.91850025],
       [ 0.94316051, -0.75198981],
       [-1.09822317, -1.01172173],
       [-0.26846436,  0.21502152],
       [ 1.54406534, -0.19460758],
       [-0.41635224,  1.53213673],
       [-1.69405546,  0.45469391],
       [-0.83391803, -1.05219369],
       [-0.47794268,  1.16802366],
       [ 0.1591679 , -0.46713415],
       [ 2.57205029, -1.26553038],
       [-1.34932831, -0.37788334],
       [-0.21635772, -0.19743135],
       [-0.81792962,  1.20379693],
       [ 1.0580857 , -0.36382599],
       [ 0.33332921,  0.28205947],
       [ 0.90855273,  0.81698089],
       [ 0.63046251,  2.10540536],
       [-1.40337141,  0.02926122],
       [ 0.537267  ,  1.27351656],
       [ 2.71015281,  0.46266549],
       [ 0.27486612, -0.18490592],
       [ 1.65476386,  0.69147326],
       [-0.40498418,

In [13]:
pca.get_covariance()

array([[1.02564103, 0.20478114, 0.080118  ],
       [0.20478114, 1.02564103, 0.19838882],
       [0.080118  , 0.19838882, 1.02564103]])

In [16]:
print(pca.explained_variance_)

[1.3536065  0.94557084]


In [19]:
print(pca.components_)

[[ 0.53875915  0.65608325  0.52848211]
 [-0.69363291 -0.01057596  0.72025103]]


In [24]:
# Ignoring arbitary signs
print(np.abs(eigen_vectors[:,:2]))
print("--------------------------")
print(np.abs(pca.components_.T))

[[0.53875915 0.69363291]
 [0.65608325 0.01057596]
 [0.52848211 0.72025103]]
--------------------------
[[0.53875915 0.69363291]
 [0.65608325 0.01057596]
 [0.52848211 0.72025103]]
